<a href="https://colab.research.google.com/github/suryasai99/Object_detection/blob/main/8_loss_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing libraries

In [1]:
import os
import requests
from zipfile import ZipFile

import torch
import numpy as np

In [2]:
def download_file(url, save_name):
    """
    "Download and save the file."

    arguments:
    url (str): URL path of the file.
    save_name: (str): file path to save the downloaded file.
    """
    file = requests.get(url)
    open(save_name, 'wb').write(file.content)
    print(f"Downloaded {save_name}...")
    return

In [3]:
def unzip(zip_file_path=None):
    """
    "Unzip the file"

    arguments:
    zip_file_path (str): The zipped file path

    """
    try:
        with ZipFile(zip_file_path) as z:
            z.extractall("./")
            print(f"Extracted {zip_file_path}...\n")
    except:
        print("Invalid file")

    return

In [4]:
if not os.path.exists('encoder_scripts'):
    download_file(
                  'https://www.dropbox.com/s/slmasud3oeodtol/loss_function.zip?dl=1',
                  'loss_function.zip'
                 )

    unzip('loss_function.zip')

Downloaded loss_function.zip...
Extracted loss_function.zip...



In [5]:
os.listdir("./")

['.config',
 'general_encoder.py',
 'detection_loss.py',
 'loss_function.zip',
 'sample_data']

In [6]:
from general_encoder import DataEncoder
from detection_loss import SmoothL1Loss, OHEMLoss

img_height = img_width = 300

num_classes = 5  # including background

bounding_boxes = torch.tensor([[100, 100, 150, 150], [120, 200, 160, 250]], dtype=torch.float)
targets = torch.tensor([2, 4], dtype=torch.float)

data_encoder = DataEncoder((img_height, img_width),
                           classes=["__background__", "person", "bicycle", "car", "motorcycle"],
                           )

bboxes, labels = data_encoder.encode(bounding_boxes, targets)

In [7]:
torch.manual_seed(21)
np.random.seed(21)

pred_boxes_np = np.random.rand(1, labels.shape[0], 4)
pred_probs_np = np.random.rand(1, labels.shape[0], num_classes)

pred_bboxes = torch.from_numpy(pred_boxes_np)
pred_prob = torch.from_numpy(pred_probs_np)

loc_loss = SmoothL1Loss()
ohem_loss = OHEMLoss(num_classes)

loss_fn = dict(
             loc_loss = SmoothL1Loss(),
             cls_loss = OHEMLoss(num_classes)
            )

In [8]:
loc_loss_weightage = 15.0
cls_loss_weightage = 0.5

reg_loss = loc_loss(pred_bboxes, bboxes.unsqueeze(0), labels.unsqueeze(0))
cls_loss = ohem_loss(pred_prob, labels.unsqueeze(0))

total_detection_loss = loc_loss_weightage * reg_loss + cls_loss_weightage * cls_loss

print(f"Bounding Box Loss:\t{reg_loss:.5}")
print(f"Classification Loss:\t{cls_loss:.5}")
print(f"Total Loss:\t{total_detection_loss:.5}")

Bounding Box Loss:	0.6226
Classification Loss:	8.6533
Total Loss:	13.666
